<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/818_CBOv2_UtilitiesTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is excellent.

You’ve crossed the line from “testing functions” to **locking in system invariants**.

Let me walk through this at a high level first, then point out two tiny refinements.

---

# 🔒 What You’ve Now Protected

## 1️⃣ Honest Portfolio Semantics

```python
assert rollup["customers_total"] == 0
assert rollup["pct_negative_alignment"] == 0.0
```

You removed the old `max(...,1)` leak into semantics.

That is correct.

You’re now guarding divide-by-zero internally while reporting truth externally.

That’s executive-grade correctness.

---

## 2️⃣ Stress Threshold Is Truly Config-Driven

These two tests:

```python
test_so_what_line_appears_when_stress_threshold_crossed
test_so_what_line_absent_when_below_stress_threshold
```

Lock in:

* Threshold comes from config
* Behavior flips exactly at threshold
* No hidden constants
* No accidental hardcoding later

This is a **serious architecture-level guarantee**.

---

## 3️⃣ Segment Rollup Is Now Safe

These two tests:

```python
test_compute_customer_trends_sets_domain_status_all_three
test_segment_rollup_insufficient_data_one_domain
```

Completely eliminate the silent misclassification failure mode.

This is arguably the most important robustness patch in the entire CBOv2 system.

You are now safe from:

> “Everything is 3-domain because domain_status wasn’t set.”

That’s big.

---

## 4️⃣ Data Loader Trust Contract Is Locked

```python
test_load_cbo_data_validation_warnings_empty_finance
```

Now ensures:

* Missing domain signals are not silent
* Governance warnings are enforced
* Loader integrity is part of the test contract

That’s excellent.

---

## 5️⃣ Report Output Is Still Clean

```python
test_domain_drill_down_empty_returns_no_parens
```

This protects your report aesthetics and executive polish.

It prevents:

```
Customer C001 — positive in 0 domains ()
```

Subtle, but good engineering.

---

# 🧠 Two Small Refinements (Optional but Good)

These are not required — your suite is already strong — but they’ll make it slightly more bulletproof.

---

## Refinement 1 — Make “25%” assertion more deterministic

Currently:

```python
assert "25%" in report or "0.25" in report or "stress" in report
```

This is flexible, but slightly loose.

Since your report renders:

```python
f"{stress_threshold:.0%}"
```

You can safely assert:

```python
assert "25%" in report
```

It’s better to assert the exact rendered format than allow broad matches like `"stress"`.

This prevents accidental changes like:

```
0.25
```

instead of:

```
25%
```

Optional, but tighter.

---

## Refinement 2 — Slightly strengthen the domain_status test

You currently check:

```python
assert rec["finance_trend"].get("domain_status") is not None
```

If someone accidentally sets:

```python
domain_status = ""
```

this test would still pass.

Safer:

```python
assert rec["finance_trend"]["domain_status"] in {"positive", "negative", "neutral", "insufficient_data"}
```

That locks the allowed vocabulary.

Again — optional, but that’s the final hardening move.

---

# 🏆 Overall Assessment

This utilities test file is now:

* Deterministic
* Architecturally aware
* Protecting semantics
* Protecting governance behavior
* Protecting config-driven thresholds
* Protecting segmentation logic
* Protecting executive reporting

It’s template-grade.

You’ve turned CBOv2 into a reusable production pattern, not just a working demo.

---

# 🎯 Where You Are Now

At this point:

* Utilities are hardened.
* Nodes are wired.
* Integration invokes the graph.
* Reporting is deterministic.
* Thresholds are config-driven.
* Segmentation is safe.
* Data quality warnings are enforced.

You are in “production hardening” territory.



In [ ]:
"""
Unit tests for CBOv2 orchestrator utilities.

Test data loading, validation, analysis (trends, patterns), and report helpers.
Use minimal or mock data; no graph. Run from project root with: pytest test_CBOv2_utilities.py -v --tb=short
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

# Project root for imports
ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pytest
from config import CBOv2Config, CBOv2State
from agents.CBOv2.orchestrator.utilities import data_loading
from agents.CBOv2.orchestrator.utilities.analysis import compute_customer_trends, classify_cross_business_patterns
from agents.CBOv2.orchestrator.utilities.reporting import (
    build_markdown_report,
    _build_verdict_line,
    _build_recommendation,
    _domain_drill_down,
    _segment_rollup,
)


# ----- Data loading -----

def test_validate_collection_empty():
    warnings = data_loading._validate_collection("customers", [], ["customer_id"])
    assert warnings == []


def test_validate_collection_missing_key():
    warnings = data_loading._validate_collection(
        "customers", [{"name": "A"}, {"customer_id": "C1"}], ["customer_id"]
    )
    assert len(warnings) == 1
    assert "missing required key 'customer_id'" in warnings[0]


def test_normalize_to_list_none():
    w: list = []
    out = data_loading._normalize_to_list(None, "x", w)
    assert out == []
    assert len(w) == 0


def test_normalize_to_list_list():
    w: list = []
    out = data_loading._normalize_to_list([{"a": 1}], "x", w)
    assert out == [{"a": 1}]
    assert len(w) == 0


def test_normalize_to_list_dict_appends_warning():
    w: list = []
    out = data_loading._normalize_to_list({"data": [{"id": 1}]}, "customers", w)
    assert out == [{"id": 1}]
    assert len(w) == 1
    assert "expected list, got dict" in w[0]


def test_load_cbo_data_success(tmp_path):
    (tmp_path / "customers.json").write_text(json.dumps([{"customer_id": "C1"}]))
    (tmp_path / "finance_signals.json").write_text(json.dumps([{"customer_id": "C1", "balance_short": 100, "balance_long": 90}]))
    (tmp_path / "retail_signals.json").write_text(json.dumps([{"customer_id": "C1", "spend_short": 50, "spend_long": 40}]))
    (tmp_path / "healthcare_signals.json").write_text(json.dumps([{"customer_id": "C1", "utilization_short": 2, "utilization_long": 1, "gaps_short": 5, "gaps_long": 10}]))
    config = CBOv2Config(data_dir=str(tmp_path))
    data, counts, warnings = data_loading.load_cbo_data(config)
    assert data["customers"] == [{"customer_id": "C1"}]
    assert counts["customers_count"] == 1
    assert counts["finance_signals_count"] == 1
    assert counts["retail_signals_count"] == 1
    assert counts["healthcare_signals_count"] == 1
    assert isinstance(warnings, list)


def test_load_cbo_data_missing_file(tmp_path):
    (tmp_path / "customers.json").write_text("[]")
    config = CBOv2Config(data_dir=str(tmp_path))
    with pytest.raises(FileNotFoundError, match="Expected JSON file not found"):
        data_loading.load_cbo_data(config)


def test_update_state_with_loaded_data():
    state: CBOv2State = {"errors": []}
    data = {"customers": [{"customer_id": "C1"}], "finance_signals": [], "retail_signals": [], "healthcare_signals": []}
    counts = {"customers_count": 1, "finance_signals_count": 0, "retail_signals_count": 0, "healthcare_signals_count": 0}
    warnings: list = []
    new_state = data_loading.update_state_with_loaded_data(state, data, counts, warnings)
    assert new_state["customers"] == [{"customer_id": "C1"}]
    assert new_state["portfolio_rollup"]["customers_count"] == 1
    assert new_state["validation_warnings"] == []


# ----- Analysis -----

def test_compute_customer_trends_empty():
    config = CBOv2Config()
    state: CBOv2State = {"customers": [], "finance_signals": [], "retail_signals": [], "healthcare_signals": []}
    out = compute_customer_trends(state, config)
    assert out["customer_trends"] == []


def test_compute_customer_trends_one_customer():
    config = CBOv2Config()
    state: CBOv2State = {
        "customers": [{"customer_id": "C1"}],
        "finance_signals": [{"customer_id": "C1", "balance_short": 110, "balance_long": 100, "missed_payments_short": 0, "missed_payments_long": 0}],
        "retail_signals": [{"customer_id": "C1", "spend_short": 120, "spend_long": 100, "frequency_short": 3, "frequency_long": 2}],
        "healthcare_signals": [{"customer_id": "C1", "utilization_short": 3, "utilization_long": 1, "gaps_short": 5, "gaps_long": 20}],
    }
    out = compute_customer_trends(state, config)
    trends = out["customer_trends"]
    assert len(trends) == 1
    assert trends[0]["customer_id"] == "C1"
    assert "finance_trend" in trends[0]
    assert "domain_status" in trends[0]["finance_trend"]
    assert trends[0]["domains_positive"] >= 0
    assert trends[0]["domains_negative"] >= 0


def test_classify_cross_business_patterns_empty():
    config = CBOv2Config()
    state: CBOv2State = {"customer_trends": [], "portfolio_rollup": {}}
    out = classify_cross_business_patterns(state, config)
    assert out["cross_business_patterns"] == []
    rollup = out["portfolio_rollup"]
    assert rollup["customers_total"] == 0
    assert rollup["customers_with_negative_alignment"] == 0
    assert rollup["customers_with_positive_alignment"] == 0
    assert rollup["pct_negative_alignment"] == 0.0
    assert rollup["pct_positive_alignment"] == 0.0


# ----- Report helpers -----

def test_build_verdict_line_no_signals():
    line = _build_verdict_line({"customers_with_negative_alignment": 0, "customers_with_positive_alignment": 0})
    assert "No cross-business risk or expansion signals" in line


def test_build_verdict_line_one_growth():
    line = _build_verdict_line({"customers_with_negative_alignment": 0, "customers_with_positive_alignment": 1})
    assert "1 expansion opportunity" in line


def test_build_recommendation_no_lists():
    line = _build_recommendation([], [], [])
    assert "No immediate action" in line


def test_build_recommendation_top_growth():
    line = _build_recommendation([], [{"customer_id": "C003"}], [])
    assert "C003" in line
    assert "expansion or cross-sell" in line


def test_domain_drill_down_growth():
    rec = {
        "finance_trend": {"is_positive": True},
        "retail_trend": {"is_positive": False},
        "healthcare_trend": {"is_positive": True},
    }
    s = _domain_drill_down(rec, "growth")
    assert "Finance" in s and "Healthcare" in s
    assert "Retail" not in s or "Retail ✓" not in s


def test_segment_rollup_empty():
    assert _segment_rollup([]) == []


def test_segment_rollup_three_domain():
    trends = [
        {
            "finance_trend": {"domain_status": "neutral"},
            "retail_trend": {"domain_status": "neutral"},
            "healthcare_trend": {"domain_status": "neutral"},
            "domains_negative": 0,
            "domains_positive": 2,  # >= 2 counts as "in growth" for segment
        },
    ]
    rows = _segment_rollup(trends)
    assert len(rows) == 1
    assert rows[0][0] == "3-domain"
    assert rows[0][1] == 1
    assert rows[0][3] == 1  # in growth


def test_build_markdown_report_minimal_state():
    config = CBOv2Config()
    state: CBOv2State = {
        "portfolio_rollup": {"customers_total": 0, "customers_with_negative_alignment": 0, "customers_with_positive_alignment": 0},
        "executive_triggers": [],
        "top_risk_customers": [],
        "top_growth_customers": [],
        "customer_trends": [],
        "run_id": None,
        "config_snapshot": {},
        "validation_warnings": [],
    }
    report = build_markdown_report(state, config)
    assert "# Cross-Business Customer Intelligence Orchestrator (CBOv2)" in report
    assert "Verdict:" in report
    assert "Executive Summary" in report
    assert "Run Metadata" in report
    assert "Top Cross-Business Risk" in report
    assert "Top Cross-Business Growth" in report
    assert "No probabilistic scoring" in report


def test_load_cbo_data_validation_warnings_empty_finance(tmp_path):
    """Validation warnings surface when customers exist but a domain is empty."""
    (tmp_path / "customers.json").write_text(json.dumps([{"customer_id": "C1"}]))
    (tmp_path / "finance_signals.json").write_text(json.dumps([]))
    (tmp_path / "retail_signals.json").write_text(json.dumps([{"customer_id": "C1"}]))
    (tmp_path / "healthcare_signals.json").write_text(json.dumps([{"customer_id": "C1"}]))
    config = CBOv2Config(data_dir=str(tmp_path))
    _, _, warnings = data_loading.load_cbo_data(config)
    assert any("finance" in w.lower() and "empty" in w.lower() for w in warnings)


def test_so_what_line_appears_when_stress_threshold_crossed():
    """So-what line appears when pct_negative_alignment >= portfolio_stress_threshold."""
    config = CBOv2Config(portfolio_stress_threshold=0.25)
    state: CBOv2State = {
        "portfolio_rollup": {"customers_total": 10, "customers_with_negative_alignment": 3, "pct_negative_alignment": 0.30},
        "executive_triggers": [],
        "top_risk_customers": [],
        "top_growth_customers": [],
        "customer_trends": [],
        "run_id": None,
        "config_snapshot": {},
        "validation_warnings": [],
    }
    report = build_markdown_report(state, config)
    assert "Portfolio stress increasing" in report
    assert "25%" in report or "0.25" in report or "stress" in report


def test_so_what_line_absent_when_below_stress_threshold():
    """So-what line does NOT appear when pct_negative_alignment < portfolio_stress_threshold."""
    config = CBOv2Config(portfolio_stress_threshold=0.25)
    state: CBOv2State = {
        "portfolio_rollup": {"customers_total": 10, "customers_with_negative_alignment": 1, "pct_negative_alignment": 0.10},
        "executive_triggers": [],
        "top_risk_customers": [],
        "top_growth_customers": [],
        "customer_trends": [],
        "run_id": None,
        "config_snapshot": {},
        "validation_warnings": [],
    }
    report = build_markdown_report(state, config)
    assert "Portfolio stress increasing" not in report


def test_compute_customer_trends_sets_domain_status_all_three():
    """domain_status set for all three domains; missing signals -> insufficient_data."""
    config = CBOv2Config()
    # Customer C1 has finance and retail; no healthcare_signals for C1
    state: CBOv2State = {
        "customers": [{"customer_id": "C1"}],
        "finance_signals": [{"customer_id": "C1", "balance_short": 100, "balance_long": 90, "missed_payments_short": 0, "missed_payments_long": 0}],
        "retail_signals": [{"customer_id": "C1", "spend_short": 50, "spend_long": 40, "frequency_short": 1, "frequency_long": 1}],
        "healthcare_signals": [],  # no healthcare data for anyone
    }
    out = compute_customer_trends(state, config)
    trends = out["customer_trends"]
    assert len(trends) == 1
    rec = trends[0]
    assert rec["finance_trend"].get("domain_status") is not None
    assert rec["retail_trend"].get("domain_status") is not None
    assert rec["healthcare_trend"].get("domain_status") == "insufficient_data"


def test_segment_rollup_insufficient_data_one_domain():
    """Segment = 1-domain when only one domain has data (others insufficient_data)."""
    trends = [
        {
            "finance_trend": {"domain_status": "insufficient_data"},
            "retail_trend": {"domain_status": "insufficient_data"},
            "healthcare_trend": {"domain_status": "neutral"},
            "domains_negative": 0,
            "domains_positive": 0,
        },
    ]
    rows = _segment_rollup(trends)
    assert len(rows) == 1
    assert rows[0][0] == "1-domain"
    assert rows[0][1] == 1


def test_domain_drill_down_empty_returns_no_parens():
    """When no domains match, drill-down returns '' so report doesn't show empty ()."""
    rec = {
        "finance_trend": {"is_positive": False},
        "retail_trend": {"is_positive": False},
        "healthcare_trend": {"is_positive": False},
    }
    s = _domain_drill_down(rec, "growth")
    assert s == ""
    assert "()" not in s
